## Source

- https://www.kaggle.com/datasets/priyamchoksi/credit-card-transactions-dataset

## Dataset Description

The Credit Card Transactions Dataset contains historical credit card transaction records, customer demographic information, merchant details, geographic locations, and fraud labels. The dataset is designed for fraud detection and transaction analysis tasks.

The main objective of this project is to analyze transaction patterns and develop a machine learning model capable of predicting whether a transaction is fraudulent or legitimate. The dataset includes both numerical and categorical features, making it suitable for data cleaning, exploratory data analysis, feature engineering, preprocessing pipelines, classification modeling, and deployment.

### Problem Type
- Binary Classification

In [ ]:
import pandas as pd
import numpy as np 
import plotly.express as px

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('credit_card_transactions.csv')
df

## Data Understanding

| Column | Description |
|----------|----------|
| Unnamed: 0 | Row index generated during dataset creation |
| trans_date_trans_time | Date and time of the transaction |
| cc_num | Credit card number used for the transaction |
| merchant | Merchant where the transaction occurred |
| category | Transaction category (shopping, grocery, entertainment, etc.) |
| amt | Transaction amount in USD |
| first | Customer's first name |
| last | Customer's last name |
| gender | Customer gender |
| street | Customer street address |
| city | Customer city |
| state | Customer state |
| zip | Customer ZIP code |
| lat | Customer latitude coordinate |
| long | Customer longitude coordinate |
| city_pop | Population of the customer's city |
| job | Customer occupation |
| dob | Customer date of birth |
| trans_num | Unique transaction identifier |
| unix_time | Transaction timestamp in Unix format |
| merch_lat | Merchant latitude coordinate |
| merch_long | Merchant longitude coordinate |
| is_fraud | Target variable indicating whether the transaction is fraudulent (1) or legitimate (0) |
| merch_zipcode | Merchant ZIP code |

### Initial Observations

- The dataset contains both numerical and categorical features.
- The target variable is `is_fraud`.
- The problem is a binary classification problem.
- Some columns contain personally identifiable information such as names and addresses.
- Geographic information is available through latitude and longitude coordinates.
- Date and time information can be used for feature engineering.
- The dataset is expected to be highly imbalanced because fraudulent transactions are usually much fewer than legitimate transactions.

## Data Exploration

In [ ]:
df.info()

- Check Inconsistent Values

In [ ]:
df.describe().round(2)

In [ ]:
df.select_dtypes(include='object').describe()

- check duplicates

In [ ]:
df.duplicated().sum()

- Check Missing Values

In [ ]:
df.isna().sum()

## Data Cleaning

In [ ]:
df.columns

- Check Feature Imbalance and Unique Values

In [ ]:
#show values for each categorical column
for col in df.select_dtypes(include = 'object').columns :

    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print(df[col].value_counts())
    print('-' * 50)

- Drop unamed column

In [ ]:
df.drop(columns = 'Unnamed: 0' , inplace = True)

In [ ]:
df.columns

- Convert datetime columns

In [ ]:
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])

- check Outliers

In [ ]:
from datasist.structdata import detect_outliers 
for col in df.select_dtypes(include = 'number').columns :
    outliers = detect_outliers(data = df , n = 0 , features = [col])
    print(col)
    print(outliers)
    print('-'*50)

## Feature Engineering

- Convert dob column to age column

In [ ]:
df['age'] = (df['trans_date_trans_time'].dt.year - df['dob'].dt.year)

df.drop(columns = 'dob' , inplace = True)


- Calculate distance between the customer location and the merchant location

In [ ]:
# distance feature
from geopy.distance import geodesic
def calculate_distance(row):
    transaction_location = (row['lat'], row['long'])
    merchant_location = (row['merch_lat'], row['merch_long'])
    distance = geodesic(transaction_location, merchant_location).kilometers
    return distance
df['distance'] = df.apply(calculate_distance, axis=1)
df.drop(columns = ['lat', 'long', 'merch_lat', 'merch_long'] , inplace = True)

- Extract Tranasaction year , month , hour and day of week from transaction date time column

In [ ]:
# time features
df['transaction_year'] = df['trans_date_trans_time'].dt.year
df['transaction_month'] = df['trans_date_trans_time'].dt.month
df['transaction_hour'] = df['trans_date_trans_time'].dt.hour
df['transaction_dayofweek'] = df['trans_date_trans_time'].dt.dayofweek
df['transaction_day'] = df['trans_date_trans_time'].dt.day


### Removed the unix_time , trasn date columns after feature engineering bec.
- It provides redundant information already captured by engineered time features.
- It does not add additional behavioral insight about user transactions.
- Keeping it may introduce noise and unnecessary complexity to the model.

In [ ]:
df.drop(columns = ['trans_date_trans_time','unix_time'] , inplace = True)

In [ ]:
# to show ther is missing values after feature engineering or not 
df.isnull().sum()

# EDA

### Univariate 

- This chart shows distribution of transaction amounts

In [ ]:
fig=px.histogram(df,x='amt',title='Distribution of Transaction Amounts',nbins=30)
fig.show()

- This chart show distribution of ages

In [ ]:
fig2=px.histogram(df,x='age',title='Distribution of Age',nbins=30)
fig2.show()

- This chart shows distribution of distance 

In [ ]:
fig3=px.histogram(df,x='distance',title='Distribution of Distance')
fig3.show()

- Distribution of is fraud

In [ ]:
fraud_counts = df['is_fraud'].value_counts().reset_index()

fig = px.bar(
    fraud_counts,
    x='is_fraud',
    y='count',
    color='is_fraud',
    title='Fraud vs Non-Fraud'
)

fig.show()

- Distribution by category transaction

In [ ]:
fig = px.bar(
    df['category'].value_counts().sort_values(ascending=False),
    title='Top 10 Categories'
)

fig.show()

- Distribution of gender

In [ ]:
# gender dist 
fig6=px.histogram(df,x='gender',title='Distribution of Gender')
fig6.show()

### Bivariate Analysis

- Are fraudulent transactions associated with higher amounts?

In [ ]:
fig1 = px.box(df, x='is_fraud', y='amt',
              title='Amount vs Fraud')
fig1.show()

- Fraudulent transactions are not necessarily associated with higher amounts. The distribution shows that high-value transactions mostly belong to non-fraud cases as outliers.

- Do fraudulent transactions occur at larger distances?

In [ ]:
fig2 = px.box(df, x='is_fraud', y='distance',
              title='Distance vs Fraud')
fig2.show()

- At what hours does fraud occur most frequently?

In [ ]:
hour_labels = {
    0:'12 AM', 1:'1 AM', 2:'2 AM', 3:'3 AM',
    4:'4 AM', 5:'5 AM', 6:'6 AM', 7:'7 AM',
    8:'8 AM', 9:'9 AM', 10:'10 AM', 11:'11 AM',
    12:'12 PM', 13:'1 PM', 14:'2 PM', 15:'3 PM',
    16:'4 PM', 17:'5 PM', 18:'6 PM', 19:'7 PM',
    20:'8 PM', 21:'9 PM', 22:'10 PM', 23:'11 PM'}

fraud_hour = (
    df[df['is_fraud'] == 1]['transaction_hour']
    .map(hour_labels)
    .value_counts())

fig = px.bar(
    x=fraud_hour.index,
    y=fraud_hour.values,
    labels={'x':'Hour', 'y':'Fraud Count'},
    title='Fraud Cases by Hour')

fig.show()

- Does fraud vary by day of the week?

In [ ]:
day_names = {
    0: 'Monday',
    1: 'Tuesday',
    2: 'Wednesday',
    3: 'Thursday',
    4: 'Friday',
    5: 'Saturday',
    6: 'Sunday'
}

fraud_days = (
    df[df['is_fraud'] == 1]['transaction_dayofweek'].map(day_names).value_counts().sort_values(ascending=False)
)

fig = px.bar(
    x=fraud_days.index,
    y=fraud_days.values,
    labels={'x':'Day of Week', 'y':'Fraud Count'},
    title='Fraud Cases by Day of Week'
)

fig.show()

- Fraud by state (top 10)

In [ ]:
px.bar(
    df.groupby('state')['is_fraud'].sum().sort_values(ascending=False).head(10),
    title='Top 10 States by Fraud Count'
).show()

- Is fraud more common among certain age groups?

In [ ]:
fig6 = px.box(df, x='is_fraud', y='age',
              title='Age vs Fraud')
fig6.show()

In [ ]:
df.columns

## Drop usless columns before ml 

-drop columns that represent identifiers that will be overfit data with ml  

In [ ]:
df.drop(columns =['cc_num','first','last','street','trans_num'],inplace=True)

- drop high cardinality columns that unnecessary in ml

In [ ]:
df.drop(columns =['merchant','job','city','zip'],inplace=True)

In [ ]:
df.columns

In [ ]:
df.to_csv('cleaned_credit_card_transactions.csv' , index = False)

# Data Preprocessing

# Data Preprocessing Steps for Machine Learning :

* 1- Split Data into Input Features and Target Column

* 2- Split Data into Train & Test

* 3- Numerical Cols : Impute Missing --> Scaling

* 4- Categorical Cols : Impute Missing --> Encoding

* 5- Handle Imbalance

### Split Data into input Features and Target Feature

In [ ]:
df2=pd.read_csv("cleaned_credit_card_transactions.csv")

In [ ]:
x = df2.drop('is_fraud', axis=1)
y = df2['is_fraud']

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
y.value_counts()  # imbalance

## Numerical pipeline

In [ ]:
num_cols = x.select_dtypes(include= 'number').columns
num_cols

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

imputer=SimpleImputer(strategy='median')
scaler=RobustScaler()


num_pipeline = Pipeline([('imputer', imputer), ('scaler', scaler )])

num_pipeline

## Categorical Pipelines

In [ ]:
from category_encoders import BinaryEncoder #state

be = BinaryEncoder()

state_pipeline = Pipeline([('BE', be)])
state_pipeline

In [ ]:
from sklearn.preprocessing import OneHotEncoder # gender,category

ohe = OneHotEncoder(drop= 'first', sparse_output= False)

ohe_pipeline = Pipeline([('OHE Pipeline', ohe)])
ohe_pipeline

### Assign each column to the corresponding Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
        ('num', num_pipeline, num_cols),
        ('state', state_pipeline, ['state']),
        ('ohe', ohe_pipeline, ['gender', 'category'])],
         remainder='drop'
)

preprocessor

### Modelling

In [ ]:
from imblearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

models = [('Logistic Regression',LogisticRegression(random_state=42,max_iter=1000,class_weight='balanced')),
          ('Decision Tree',DecisionTreeClassifier(random_state=42)),
          ('Random Forest',RandomForestClassifier( random_state=42,class_weight='balanced')),
          ('XGBoost',XGBClassifier(random_state=42,eval_metric='logloss')),
          ('CatBoost', CatBoostClassifier(random_state=42,verbose=0)) ]

for model in models:

    model_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('Model', model[1])
    ])
    result = cross_validate(model_pipeline,x_train, y_train,scoring='f1', cv=3 ,return_train_score=True, n_jobs=-1,error_score='raise')

    print(model[0])
    print('Train Score :', round(result['train_score'].mean() * 100, 2))
    print('Test Score  :', round(result['test_score'].mean() * 100, 2))
    print('-' * 50)

- Multiple classification models were evaluated. CatBoost achieved the highest F1-score and was selected as the final mode
- Class balancing techniques were tested but did not improve the F1 score, so the default CatBoost configuration was retained.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV # tuning for reach best accuracy 

pipeline = Pipeline([('preprocessor', preprocessor),
                     ('Model', CatBoostClassifier(verbose=0,random_state=42))
])
params = {
    'Model__depth': [4, 6, 8, 10],
    'Model__learning_rate': [0.01, 0.05, 0.1],
    'Model__iterations': [100, 200, 300]
}
search = RandomizedSearchCV(pipeline , param_distributions=params  ,n_iter=5,
                            scoring='f1', # evaluation metric cv=3,
                            n_jobs=1,
                            random_state=42
)
search.fit(x_train, y_train)

In [ ]:
search.best_params_

In [ ]:
search.best_score_

- RandomizedSearchCV was applied to optimize the CatBoost hyperparameters. However, the tuned model did not outperform the baseline model (F1-score: 88.54% vs. 88.44%). Therefore, the baseline CatBoost model was retained as the final model.

In [ ]:
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('Model', CatBoostClassifier(verbose=0,random_state=42 ))
])

final_pipeline.fit(x_train, y_train)

- The dataset was highly imbalanced. Instead of relying on accuracy, I used F1-score as the primary evaluation metric because it is more suitable for imbalanced classification problems. I did not apply resampling techniques since CatBoost achieved a strong F1-score of 88.54% without them.

In [ ]:
y_pred = final_pipeline.predict(x_test)
y_pred

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

- The dataset was highly imbalanced, with fraudulent transactions representing less than 1% of all records.

- Several imbalance handling techniques were considered. However, the baseline CatBoost model achieved strong performance without resampling,      obtaining:
- Precision: 96%
- Recall: 85%
- F1-Score: 90%

Therefore, SMOTE was not applied in the final model, as the model already demonstrated excellent fraud detection capability while maintaining high precision.

In [ ]:
final_pipeline.fit(x, y)

### Feature Importance

In [ ]:
feature_names = search.best_estimator_.named_steps['preprocessor'].get_feature_names_out()

importances = search.best_estimator_.named_steps['Model'].feature_importances_

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

importance_df.head(10)

- Feature Importance analysis showed that transaction amount (amt), transaction hour, and customer age were the most influential features in fraud prediction. Transaction category also played a significant role, particularly gas_transport, grocery_pos, and shopping_pos transactions. This suggests that both transaction characteristics and customer behavior patterns contribute substantially to identifying fraudulent activities.

In [ ]:
import plotly.express as px

top10 = importance_df.head(10)

fig = px.bar(
    top10,
    x='Feature',
    y='Importance',
    title='Top 10 Important Features'
)

fig.show()

In [ ]:
import joblib

joblib.dump(final_pipeline, "fraud_model_NEW.pkl")


### Deployment

In [ ]:
df2.to_parquet("dashboard_data.parquet", index=False)

In [ ]:
# ! streamlit run streamlit_app.py